# HMM directly on subtrial data (single-session pilot)

Motivation: `cat-HMM.ipynb` fits an HMM on the *discrete watershed trial-cluster* sequence - a lossy two-stage pipeline (subtrial data -> discretize into ~4-18 trial modes -> HMM on the discrete label sequence) that, after fixing its bugs and testing properly, showed no robust engagement-like structure at full scale. This notebook skips the discretization step entirely: fit a Gaussian-emission HMM directly on a continuous, dimensionality-reduced representation of each trial's raw subtrial (syllable) sequence.

Dimensionality reduction (PCA) is fit **globally**, on a large pooled sample of trials across many sessions, not per-session - otherwise "PC1" wouldn't mean the same thing from one session to the next, and no cross-session comparison would be possible later. Every session's trials (including the one session used here) are projected through that same fixed global transform.

This is a first pilot on **one session only**, to check the mechanics work and see what the two states look like, before deciding whether to scale up.

In [ ]:
"""
IMPORTS
"""
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA
from hmmlearn import hmm


In [ ]:
prefix = '/home/ines/repositories/representation_learning_variability/paper-individuality/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/'


def first_existing(*relative_paths):
    for rel in relative_paths:
        if os.path.exists(prefix + rel):
            return rel
    raise FileNotFoundError(f'None of {relative_paths} found under {prefix}')


SEQUENCES_FILE = first_existing(
    '1_segmentation/all_sequences_26-03-2026',
    'clustering/all_sequences_26-03-2026',
)
print(f'Using SEQUENCES_FILE = {SEQUENCES_FILE}')

all_sequences = pd.read_parquet(prefix + SEQUENCES_FILE)
all_sequences['trial_id'] = all_sequences['sample'].str.split().str[1:2].str.join('').astype(float)
all_sequences['session'] = all_sequences['sample'].str.split().str[0:1].str.join('')
print(f"{len(all_sequences)} rows, {all_sequences['sample'].nunique()} trials, "
      f"{all_sequences['session'].nunique()} sessions")


## Build the per-trial feature vector

Same one-hot encoding `clustering_trial_syllables_watershed.ipynb` uses before its own PCA step: each trial has 4 epochs (Pre-quiescence / Quiescence / Choice / ITI) x 10 time-bins = 40 raw syllable codes (0-31, encoding paw-state + 8*whisking + 16*licking - see `engagement_trial_modes.ipynb`'s fingerprint section for the same encoding). Each of the 40 (epoch, bin) positions is one-hot encoded over the 32 possible codes, giving a 1280-dim binary vector per trial. Trials missing an epoch, or with any NaN code, are dropped.

In [ ]:
EPOCHS = ['Pre-quiescence', 'Quiescence', 'Choice', 'ITI']
N_BINS = 10
N_CODES = 32  # raw binned_sequence codes are 0-31


def build_design_matrix(sub_seq):
    """One row per trial, one-hot encoded across all 4*10=40 (epoch, bin) positions x 32 codes."""
    wide = sub_seq.pivot_table(index='sample', columns='broader_label', values='binned_sequence', aggfunc='first')
    wide = wide.dropna(subset=EPOCHS)
    n = len(wide)
    X = np.zeros((n, len(EPOCHS) * N_BINS * N_CODES), dtype=np.float32)
    valid = np.ones(n, dtype=bool)
    for e, epoch in enumerate(EPOCHS):
        arr = np.vstack(wide[epoch].values)
        bad = np.isnan(arr).any(axis=1)
        valid &= ~bad
        codes = np.clip(np.nan_to_num(arr, nan=0).astype(int), 0, N_CODES - 1)
        for b in range(N_BINS):
            X[np.arange(n), (e * N_BINS + b) * N_CODES + codes[:, b]] = 1.0
    return wide.index[valid].values, X[valid]


## Fit PCA globally, on a large pooled sample of trials

30 components is a deliberately modest choice for this single-session pilot: with only ~500-1000 trials in one session, a Gaussian HMM with a much higher-dimensional emission (e.g. the 135 components needed for 80% variance) would have far more parameters than one session can reasonably constrain - a real overfitting risk. 30 components explains ~53% variance on a 30k-trial sample; this is an easy constant to revisit once scaling past a single session.

In [ ]:
N_PCA_COMPONENTS = 30
PCA_FIT_SAMPLE_SIZE = 30000

rng = np.random.default_rng(0)
pca_fit_sample_ids = rng.choice(all_sequences['sample'].unique(),
                                 size=min(PCA_FIT_SAMPLE_SIZE, all_sequences['sample'].nunique()),
                                 replace=False)
_, X_pca_fit = build_design_matrix(all_sequences[all_sequences['sample'].isin(pca_fit_sample_ids)])

pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0)
pca.fit(X_pca_fit)
print(f"PCA fit on {X_pca_fit.shape[0]} trials | {N_PCA_COMPONENTS} components explain "
      f"{np.cumsum(pca.explained_variance_ratio_)[-1]:.1%} variance")


## Pick one session, project its trials through the (fixed) global PCA

In [ ]:
# The first arbitrary session picked here turned out to be ~99% engaged (1051/1065 trials), making
# any engagement-overlap check on it meaningless (no real label variance to detect against). This one
# was chosen instead for having a genuinely balanced engaged/disengaged mix (53.5% disengaged, out of
# sessions with >=400 trials that are also present in 17_trial_waterclust for the discrete comparison).
EXAMPLE_SESSION = '0841d188-8ef2-4f20-9828-76a94d5343a4'
print(f"Example session: {EXAMPLE_SESSION}")

session_seq = all_sequences[all_sequences['session'] == EXAMPLE_SESSION]
trial_samples, X_session = build_design_matrix(session_seq)
X_session_pca = pca.transform(X_session)

session_trial_ids = np.array([float(s.split()[1]) for s in trial_samples])
order = np.argsort(session_trial_ids)
X_session_pca = X_session_pca[order]
session_trial_ids = session_trial_ids[order]
trial_samples = trial_samples[order]
print(f"{X_session_pca.shape[0]} trials with a complete feature vector, sorted into chronological order")


In [ ]:
def first_existing_any(base, *relative_paths):
    for rel in relative_paths:
        if os.path.exists(base + rel):
            return rel
    raise FileNotFoundError(f'None of {relative_paths} found under {base}')


TRIAL_FILE = first_existing_any(prefix, 'GLM-HMM/data/all_trials_06-07-2026', '4_mice/all_trials_04-05-2026')
print(f'Using TRIAL_FILE = {TRIAL_FILE}')

states_df = pd.read_parquet(prefix + 'GLM-HMM/merged_behavioral_and_states.pqt')
states_df = states_df[states_df['eid'] == EXAMPLE_SESSION].copy()
right_correct = states_df['contrastLeft'].isna() & (states_df['rewarded'] == 1)
right_incorrect = states_df['contrastRight'].isna() & (states_df['rewarded'] == -1)
states_df['choice_right'] = (right_correct | right_incorrect).astype(int)
states_df['block'] = states_df['probabilityLeft']
states_df['dominant_state'] = np.where(states_df['p_state1'] >= 0.5, 'engaged', 'disengaged')

# BUGFIX (same lesson as cat-HMM.ipynb/engagement_trial_modes.ipynb): states_df has no trial_id of
# its own - only row order within the session. A naive cumcount() could silently misalign against
# the segmentation-derived trial_id used elsewhere (trial_modes, all_sequences), so recover it the
# validated way: assume the Nth row is raw trial index N, then check block/contrast/correct
# agreement against TRIAL_FILE before trusting it.
trials_full = pd.read_parquet(prefix + TRIAL_FILE)
trials_full = trials_full[trials_full['session'] == EXAMPLE_SESSION].sort_values('trial_id')

states_df = states_df.reset_index(drop=True)
states_df['pos'] = np.arange(len(states_df))
merged = states_df.merge(
    trials_full[['session', 'trial_id', 'block', 'contrast', 'correct', 'reaction']]
              .rename(columns={'block': 'block_chk', 'contrast': 'contrast_chk', 'correct': 'correct_chk'}),
    left_on='pos', right_on='trial_id', how='left')

found = merged['contrast_chk'].notna()
agree = ((merged['probabilityLeft'] == merged['block_chk'])
         & np.isclose(merged['signed_contrast'].abs(), merged['contrast_chk'].fillna(-1))
         & ((merged['rewarded'] == 1) == (merged['correct_chk'] == 1)))
agreement = agree[found].mean() if found.any() else 0
print(f'trial_id alignment agreement for this session: {agreement:.3f} ({found.sum()} trials checked)')
if agreement < 0.99:
    raise RuntimeError('trial_id alignment failed validation for this session - do not trust the merge')

states_df = merged
RT_MIN, RT_MAX = 0, 2
states_df['reaction_time'] = states_df['reaction'].where(states_df['reaction'].between(RT_MIN, RT_MAX))
print(f"{len(states_df)} trials with choice/contrast/block; "
      f"{states_df['reaction_time'].notna().sum()} with a valid RT")


In [ ]:
from scipy.optimize import curve_fit

BLOCK_COLORS = {0.2: 'tab:blue', 0.5: 'tab:orange', 0.8: 'tab:green'}


def sigmoid(x, mu, sigma, gamma, lambda_):
    return gamma + (1 - gamma - lambda_) * (1 / (1 + np.exp(-(x - mu) / sigma)))


def plot_psychometric(ax, data, title):
    for block, color in BLOCK_COLORS.items():
        block_data = data[data['block'] == block]
        if block_data.empty:
            continue
        summary = block_data.groupby('signed_contrast')['choice_right'].mean().reset_index()
        ax.scatter(summary['signed_contrast'] * 100, summary['choice_right'], color=color, alpha=0.6, s=25)
        try:
            popt, _ = curve_fit(
                sigmoid, block_data['signed_contrast'] * 100, block_data['choice_right'],
                p0=[0, 10, 0.05, 0.05], bounds=([-100, 0.1, 0, 0], [100, 100, 0.4, 0.4]),
            )
            x_range = np.linspace(-100, 100, 100)
            ax.plot(x_range, sigmoid(x_range, *popt), color=color, label=f'Block {block}')
        except Exception:
            ax.plot(summary['signed_contrast'] * 100, summary['choice_right'], color=color, alpha=0.4)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Signed contrast (%)')
    ax.set_ylabel('P(choice = right)')
    ax.legend(fontsize=7)


def plot_chronometric(ax, data, title):
    for block, color in BLOCK_COLORS.items():
        block_data = data[data['block'] == block].dropna(subset=['reaction_time'])
        if block_data.empty:
            continue
        summary = (block_data.assign(abs_contrast=block_data['signed_contrast'].abs())
                             .groupby('abs_contrast')['reaction_time'].median().reset_index())
        ax.plot(summary['abs_contrast'] * 100, summary['reaction_time'], color=color, marker='o',
                label=f'Block {block}')
    ax.set_xlabel('Contrast (%)')
    ax.set_ylabel('Median RT (s)')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)


## Fit a 2-state Gaussian HMM, with best-of-N restarts

Same lesson as `cat-HMM.ipynb`: a single EM run is prone to poor local optima. Restarts (and gap-aware `lengths`, in case any trials were dropped mid-session by the NaN filter above) are applied from the start here rather than discovered the hard way again.

In [ ]:
def contiguous_run_lengths(trial_ids_sorted):
    trial_ids_sorted = np.asarray(trial_ids_sorted)
    if len(trial_ids_sorted) == 0:
        return np.array([], dtype=int)
    gap = np.diff(trial_ids_sorted) != 1
    breaks = np.where(gap)[0] + 1
    run_starts = np.concatenate(([0], breaks))
    run_ends = np.concatenate((breaks, [len(trial_ids_sorted)]))
    return (run_ends - run_starts).astype(int)


def fit_best_of_n_restarts(X, lengths, n_components, n_restarts=10, n_iter=100, tol=1e-4,
                            covariance_type='diag', base_seed=0):
    best_model, best_score = None, -np.inf
    for seed in range(base_seed, base_seed + n_restarts):
        candidate = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                                     n_iter=n_iter, tol=tol, random_state=seed)
        try:
            candidate.fit(X, lengths=lengths)
            score = candidate.score(X, lengths=lengths)
        except Exception:
            continue
        if score > best_score:
            best_model, best_score = candidate, score
    return best_model, best_score


lengths = contiguous_run_lengths(session_trial_ids)
model, best_score = fit_best_of_n_restarts(X_session_pca, lengths, n_components=2, n_restarts=10)
hidden_states = model.predict(X_session_pca, lengths=lengths)

print(f"best of 10 restarts, log-likelihood = {best_score:.1f}")
print(f"state counts: {np.bincount(hidden_states)}")
print(f"degenerate (single state)? {len(np.unique(hidden_states)) < 2}")


## What do the two states look like?

Three views: the state sequence over trials, the same trials' first two PCA dimensions colored by state (a quick check of whether the states correspond to separable regions in feature space), and the syllable-sequence fingerprint per state (same style as `engagement_trial_modes.ipynb` and `cat-HMM.ipynb`).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 3.5), gridspec_kw={'height_ratios': [1, 2]})

state_cmap = ListedColormap(['#0173B2', '#DE8F05'])
axes[0].imshow(hidden_states[None, :], aspect='auto', cmap=state_cmap, vmin=0, vmax=1)
axes[0].set_yticks([])
axes[0].set_title(f'Session {EXAMPLE_SESSION[:8]}... | state sequence (n={len(hidden_states)} trials)',
                   fontsize=9, loc='left')
axes[0].set_xticks([])

for state, color in zip([0, 1], ['#0173B2', '#DE8F05']):
    mask = hidden_states == state
    axes[1].scatter(X_session_pca[mask, 0], X_session_pca[mask, 1], color=color, alpha=0.6, s=20,
                     label=f'state {state} (n={mask.sum()})')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=8)
axes[1].set_title('Trials in PCA space, colored by fitted state', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
RAW_CODE_TO_LABEL = {
    0: '000', 1: '100', 2: '200', 3: '300', 4: '400', 5: '500', 6: '600', 7: '700',
    8: '010', 9: '110', 10: '210', 11: '310', 12: '410', 13: '510', 14: '610', 15: '710',
    16: '001', 17: '101', 18: '201', 19: '301', 20: '401', 21: '501', 22: '601', 23: '701',
    24: '011', 25: '111', 26: '211', 27: '311', 28: '411', 29: '511', 30: '611', 31: '711',
}
LABEL_TO_GROUPED_CODE = {
    '000': 0, '010': 1, '001': 2, '011': 3,
    '100': 4, '110': 5, '101': 6, '111': 7,
    '200': 8, '210': 9, '201': 10, '211': 11,
    '300': 12, '310': 13, '301': 14, '311': 15,
    '400': 16, '410': 17, '401': 18, '411': 19,
    '500': 20, '510': 21, '501': 22, '511': 23,
    '600': 24, '610': 25, '601': 26, '611': 27,
    '700': 28, '710': 29, '701': 30, '711': 31,
}
RAW_TO_GROUPED = {code: LABEL_TO_GROUPED_CODE[label] for code, label in RAW_CODE_TO_LABEL.items()}


def raw_to_grouped_code(raw_vals):
    raw_vals = np.asarray(raw_vals, dtype=float)
    out = np.full(raw_vals.shape, np.nan)
    valid = ~np.isnan(raw_vals)
    out[valid] = [RAW_TO_GROUPED[int(v)] for v in raw_vals[valid]]
    return out


set3_colors = sns.color_palette('Set3', n_colors=20)
set3_no_yellow_grey = [c for i, c in enumerate(set3_colors) if i not in [1, 8, 9, 10]]
set3_filtered = set3_no_yellow_grey[:8]
set3_filtered = [set3_filtered[0], set3_filtered[7]] + set3_filtered[1:7]


def create_grouped_gradient_palette(n_groups=8, shades_per_group=4, base_palette='Set1'):
    base_colors = sns.color_palette(base_palette, n_colors=n_groups)

    def generate_shades(color, n_shades):
        color_rgb = np.array(mcolors.to_rgb(color))
        factors = np.linspace(0.5, 1.5, n_shades)
        return [mcolors.to_hex(color_rgb * factor + (1 - factor)) for factor in factors]

    full_palette = []
    for color in base_colors:
        full_palette.extend(generate_shades(color, shades_per_group))
    return ListedColormap(full_palette)


paw_state_palette = create_grouped_gradient_palette(n_groups=8, shades_per_group=4, base_palette=set3_filtered)


def build_mouse_seq(data, epochs=EPOCHS, n_bins=N_BINS):
    bound = int(len(data) / len(epochs))
    if bound == 0:
        return None
    mat = np.full((bound, n_bins * len(epochs)), np.nan)
    for e, epoch in enumerate(epochs):
        epoch_data = np.vstack(data.loc[data['broader_label'] == epoch, 'binned_sequence'].values)[:bound, :]
        mat[:epoch_data.shape[0], n_bins * e:n_bins * (e + 1)] = raw_to_grouped_code(epoch_data)
    return np.sort(mat, axis=0)


state_by_sample = dict(zip(trial_samples, hidden_states))
session_seq_labeled = session_seq[session_seq['sample'].isin(state_by_sample)].copy()
session_seq_labeled['hmm_state'] = session_seq_labeled['sample'].map(state_by_sample)

fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharex=True)
for ax, state in zip(axes, [0, 1]):
    mat = build_mouse_seq(session_seq_labeled[session_seq_labeled['hmm_state'] == state])
    if mat is None:
        ax.axis('off')
        continue
    ax.imshow(mat, aspect='auto', cmap=paw_state_palette, vmin=0, vmax=31, interpolation=None)
    for x in (10, 20, 30):
        ax.axvline(x=x, color='k', linestyle='--', linewidth=0.5)
    ax.set_title(f'State {state} (n={mat.shape[0]} trials)', fontsize=9)
    ax.set_xticks([10, 20, 30])
    ax.set_xticklabels(['Quiescence', 'Stimulus', 'Response'], rotation=30, ha='right', fontsize=7)
axes[0].set_ylabel('Trials (sorted)')
plt.suptitle(f'Session {EXAMPLE_SESSION[:8]}... - syllable fingerprint by subtrial-HMM state', fontsize=10)
plt.tight_layout()
plt.show()


### Psychometric and chronometric curves for the continuous subtrial-PCA states

In [ ]:
pca_state_by_trial = pd.DataFrame({'trial_id': session_trial_ids, 'hmm_state': hidden_states})
pca_session_trials = pca_state_by_trial.merge(states_df, on='trial_id', how='inner')
print(f"{len(pca_session_trials)} trials with both a subtrial-PCA-HMM state and choice/contrast/block/RT")

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for col, state in enumerate([0, 1]):
    state_data = pca_session_trials[pca_session_trials['hmm_state'] == state]
    plot_psychometric(axes[0, col], state_data, f'State {state} (n={len(state_data)}) - psychometric')
    plot_chronometric(axes[1, col], state_data, f'State {state} (n={len(state_data)}) - chronometric')
plt.suptitle(f'Session {EXAMPLE_SESSION[:8]}... - continuous subtrial-PCA HMM states', fontsize=10)
plt.tight_layout()
plt.show()


## Same session, lower dimensionality: 18 components and 2 components

Two more points on the same continuous-PCA approach, same session, same global PCA-fitting sample: 18 components (matching the number of categories the discrete trial-cluster HMM uses) and 2 components (matching the fact that `17_trial_waterclust`'s discrete clusters themselves came from a 2D UMAP embedding - so 2 components here is the closest continuous analogue of *that* pipeline's actual information bottleneck, one step upstream of the watershed discretization). Refitting PCA at each dimensionality (not just truncating the 30-component one) since PCA components are nested but each n_components fit is deterministic and cheap.

In [ ]:
def run_dimensionality_variant(n_components):
    pca_variant = PCA(n_components=n_components, random_state=0)
    pca_variant.fit(X_pca_fit)
    cum_var = np.cumsum(pca_variant.explained_variance_ratio_)[-1]
    X_variant_pca = pca_variant.transform(X_session)[order]
    lengths_variant = contiguous_run_lengths(session_trial_ids)
    model_variant, score_variant = fit_best_of_n_restarts(X_variant_pca, lengths_variant,
                                                           n_components=2, n_restarts=10)
    states_variant = model_variant.predict(X_variant_pca, lengths=lengths_variant)
    print(f"n_components={n_components} ({cum_var:.1%} variance) | log-likelihood={score_variant:.1f} | "
          f"state counts={np.bincount(states_variant)} | "
          f"degenerate={len(np.unique(states_variant)) < 2}")
    return X_variant_pca, states_variant


def plot_variant_results(X_variant_pca, states_variant, label):
    # 1. State sequence + first-two-dimensions scatter
    fig, axes = plt.subplots(2, 1, figsize=(14, 3.5), gridspec_kw={'height_ratios': [1, 2]})
    axes[0].imshow(states_variant[None, :], aspect='auto', cmap=state_cmap, vmin=0, vmax=1)
    axes[0].set_yticks([])
    axes[0].set_xticks([])
    axes[0].set_title(f'{label} | state sequence (n={len(states_variant)})', fontsize=9, loc='left')
    for state, color in zip([0, 1], ['#0173B2', '#DE8F05']):
        mask = states_variant == state
        axes[1].scatter(X_variant_pca[mask, 0], X_variant_pca[mask, 1], color=color, alpha=0.6, s=20,
                         label=f'state {state} (n={mask.sum()})')
    axes[1].set_xlabel('Dim 1')
    axes[1].set_ylabel('Dim 2')
    axes[1].legend(fontsize=8)
    axes[1].set_title(f'{label}: trials in reduced space, colored by state', fontsize=9)
    plt.tight_layout()
    plt.show()

    # 2. Syllable fingerprint by state
    state_by_sample_variant = dict(zip(trial_samples, states_variant))
    seq_labeled = session_seq[session_seq['sample'].isin(state_by_sample_variant)].copy()
    seq_labeled['hmm_state'] = seq_labeled['sample'].map(state_by_sample_variant)
    fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharex=True)
    for ax, state in zip(axes, [0, 1]):
        mat = build_mouse_seq(seq_labeled[seq_labeled['hmm_state'] == state])
        if mat is None:
            ax.axis('off')
            continue
        ax.imshow(mat, aspect='auto', cmap=paw_state_palette, vmin=0, vmax=31, interpolation=None)
        for x in (10, 20, 30):
            ax.axvline(x=x, color='k', linestyle='--', linewidth=0.5)
        ax.set_title(f'State {state} (n={mat.shape[0]} trials)', fontsize=9)
        ax.set_xticks([10, 20, 30])
        ax.set_xticklabels(['Quiescence', 'Stimulus', 'Response'], rotation=30, ha='right', fontsize=7)
    axes[0].set_ylabel('Trials (sorted)')
    plt.suptitle(f'{label} - syllable fingerprint by state', fontsize=10)
    plt.tight_layout()
    plt.show()

    # 3. Psychometric and chronometric curves by state
    state_by_trial = pd.DataFrame({'trial_id': session_trial_ids, 'hmm_state': states_variant})
    trials_variant = state_by_trial.merge(states_df, on='trial_id', how='inner')
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for col, state in enumerate([0, 1]):
        state_data = trials_variant[trials_variant['hmm_state'] == state]
        plot_psychometric(axes[0, col], state_data, f'State {state} (n={len(state_data)}) - psychometric')
        plot_chronometric(axes[1, col], state_data, f'State {state} (n={len(state_data)}) - chronometric')
    plt.suptitle(f'{label} - psychometric & chronometric', fontsize=10)
    plt.tight_layout()
    plt.show()


### 18 components

In [ ]:
X_pca18, states_18 = run_dimensionality_variant(18)
plot_variant_results(X_pca18, states_18, 'n_components=18')


### 2 components

In [ ]:
X_pca2, states_2 = run_dimensionality_variant(2)
plot_variant_results(X_pca2, states_2, 'n_components=2')


## Comparison: the same session, fit with the raw 18-cluster discrete trial modes

Same session as above (`EXAMPLE_SESSION`), same restart-fitting discipline, but now using `17_trial_waterclust`'s raw 18-category `trial_cluster` directly (no manual 4-category collapse - that collapse is exactly what wasn't wanted). This gives a direct, apples-to-apples comparison against the continuous subtrial-PCA approach above: same session, same illustrations (psychometric, chronometric, fingerprint), two different representations of the same underlying subtrial data.

In [ ]:
cluster_path = prefix + 'clustering/'
trial_modes = pd.read_parquet(cluster_path + '17_trial_waterclust')
trial_modes['trial_id'] = trial_modes['sample'].str.split().str[1:2].str.join('').astype(float)
trial_modes = trial_modes.sort_values(['session', 'trial_id']).reset_index(drop=True)

ALL_CLUSTERS = sorted(trial_modes['trial_cluster'].dropna().unique())
print(f"{len(ALL_CLUSTERS)} raw trial clusters: {ALL_CLUSTERS}")

session_modes = trial_modes.loc[trial_modes['session'] == EXAMPLE_SESSION].dropna(subset=['trial_cluster']).copy()
session_modes['trial_cluster_0idx'] = session_modes['trial_cluster'].astype(int) - 1
print(f"{len(session_modes)} trials with a trial_cluster in {EXAMPLE_SESSION[:8]}...")


In [ ]:
def fit_best_of_n_restarts_categorical(X, lengths, n_components, n_restarts=10, n_iter=150, tol=1e-4, base_seed=0):
    best_model, best_score = None, -np.inf
    for seed in range(base_seed, base_seed + n_restarts):
        candidate = hmm.CategoricalHMM(n_components=n_components, n_iter=n_iter, tol=tol, random_state=seed)
        try:
            candidate.fit(X, lengths=lengths)
            score = candidate.score(X, lengths=lengths)
        except Exception:
            continue
        if score > best_score:
            best_model, best_score = candidate, score
    return best_model, best_score


X_modes = session_modes['trial_cluster_0idx'].values.reshape(-1, 1)
lengths_modes = contiguous_run_lengths(session_modes['trial_id'].values)

modes_model, modes_score = fit_best_of_n_restarts_categorical(X_modes, lengths_modes, n_components=2)
modes_hidden_states = modes_model.predict(X_modes, lengths=lengths_modes)
session_modes['hmm_state'] = modes_hidden_states

print(f"best of 10 restarts, log-likelihood = {modes_score:.1f}")
print(f"state counts: {np.bincount(modes_hidden_states)}")
print(f"degenerate (single state)? {len(np.unique(modes_hidden_states)) < 2}")


### Psychometric and chronometric curves, this session, by discrete-18-cluster HMM state

In [ ]:
session_modes_trials = session_modes[['trial_id', 'hmm_state']].merge(states_df, on='trial_id', how='inner')
print(f"{len(session_modes_trials)} trials with both an 18-cluster-HMM state and choice/contrast/block/RT")

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for col, state in enumerate([0, 1]):
    state_data = session_modes_trials[session_modes_trials['hmm_state'] == state]
    plot_psychometric(axes[0, col], state_data, f'State {state} (n={len(state_data)}) - psychometric')
    plot_chronometric(axes[1, col], state_data, f'State {state} (n={len(state_data)}) - chronometric')
plt.suptitle(f'Session {EXAMPLE_SESSION[:8]}... - 18-cluster discrete HMM states', fontsize=10)
plt.tight_layout()
plt.show()


### Syllable fingerprint, this session, by discrete-18-cluster HMM state

In [ ]:
state_by_sample_modes = dict(zip(session_modes['sample'], session_modes['hmm_state']))
session_seq_modes_labeled = session_seq[session_seq['sample'].isin(state_by_sample_modes)].copy()
session_seq_modes_labeled['hmm_state'] = session_seq_modes_labeled['sample'].map(state_by_sample_modes)

fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharex=True)
for ax, state in zip(axes, [0, 1]):
    mat = build_mouse_seq(session_seq_modes_labeled[session_seq_modes_labeled['hmm_state'] == state])
    if mat is None:
        ax.axis('off')
        continue
    ax.imshow(mat, aspect='auto', cmap=paw_state_palette, vmin=0, vmax=31, interpolation=None)
    for x in (10, 20, 30):
        ax.axvline(x=x, color='k', linestyle='--', linewidth=0.5)
    ax.set_title(f'State {state} (n={mat.shape[0]} trials)', fontsize=9)
    ax.set_xticks([10, 20, 30])
    ax.set_xticklabels(['Quiescence', 'Stimulus', 'Response'], rotation=30, ha='right', fontsize=7)
axes[0].set_ylabel('Trials (sorted)')
plt.suptitle(f'Session {EXAMPLE_SESSION[:8]}... - syllable fingerprint by 18-cluster-HMM state', fontsize=10)
plt.tight_layout()
plt.show()


## Do any of these states overlap with true (choice-derived) engagement state?

Same permutation-invariant metrics as `cat-HMM.ipynb`'s engagement comparison (Adjusted Rand Index and best-match accuracy), applied to every subtrial-HMM variant fit on this session so far: continuous PCA at 30/18/2 components, and the discrete 18-cluster HMM. `states_df` (loaded above, with its validated `trial_id`) already has `dominant_state`.

In [ ]:
from sklearn.metrics import adjusted_rand_score


def best_match_accuracy(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return max(np.mean(a == b), np.mean(a != b))


def overlap_with_engagement(trial_ids, hmm_states, label):
    local = pd.DataFrame({'trial_id': trial_ids, 'hmm_state': hmm_states})
    merged = local.merge(states_df[['trial_id', 'dominant_state']], on='trial_id', how='inner')
    true_binary = (merged['dominant_state'] == 'engaged').astype(int).values
    hmm_binary = merged['hmm_state'].values
    if len(np.unique(hmm_binary)) < 2 or len(np.unique(true_binary)) < 2:
        print(f"{label}: n={len(merged)} | not enough state variety for a meaningful comparison")
        return
    ari = adjusted_rand_score(true_binary, hmm_binary)
    acc = best_match_accuracy(true_binary, hmm_binary)
    print(f"{label}: n={len(merged)} | ARI={ari:.3f} | best-match accuracy={acc:.3f}")


overlap_with_engagement(session_trial_ids, hidden_states, 'Continuous PCA (30 components)')
overlap_with_engagement(session_trial_ids, states_18, 'Continuous PCA (18 components)')
overlap_with_engagement(session_trial_ids, states_2, 'Continuous PCA (2 components)')
overlap_with_engagement(session_modes['trial_id'].values, session_modes['hmm_state'].values,
                         'Discrete 18-cluster HMM')
print(f"\n(for reference) engaged/disengaged split this session: "
      f"{states_df['dominant_state'].value_counts().to_dict()}")


## Model comparison at a fixed dimensionality (18 components): k=2, 3, 4

Same session, same `X_pca18` features, varying only the number of HMM states. Compared by log-likelihood, AIC, and BIC (all computed on the same fitting data - this is an in-sample comparison, not cross-validated, so treat it as a rough guide rather than a definitive answer). Diagonal-covariance `GaussianHMM` parameter count: `k^2 - 1 + 2*k*d` (transition matrix + start probabilities + per-state means + per-state diagonal variances, for `k` states and `d=18` dimensions).

In [ ]:
def n_gaussian_hmm_params(n_states, n_features, covariance_type='diag'):
    n_transmat = n_states * (n_states - 1)
    n_startprob = n_states - 1
    n_means = n_states * n_features
    n_cov = n_states * n_features if covariance_type == 'diag' else n_states * n_features * (n_features + 1) // 2
    return n_transmat + n_startprob + n_means + n_cov


model_comparison_rows = []
k_models = {2: (X_pca18, states_18)}  # k=2 already fit above; refit here just for a consistent record
for k in [2, 3, 4]:
    lengths_18 = contiguous_run_lengths(session_trial_ids)
    model_k, score_k = fit_best_of_n_restarts(X_pca18, lengths_18, n_components=k, n_restarts=10)
    states_k = model_k.predict(X_pca18, lengths=lengths_18)
    k_models[k] = (model_k, states_k)
    n_params = n_gaussian_hmm_params(k, X_pca18.shape[1])
    n_obs = X_pca18.shape[0]
    aic = -2 * score_k + 2 * n_params
    bic = -2 * score_k + n_params * np.log(n_obs)
    model_comparison_rows.append({'k': k, 'log_likelihood': score_k, 'n_params': n_params,
                                   'AIC': aic, 'BIC': bic,
                                   'degenerate': len(np.unique(states_k)) < 2,
                                   'state_counts': np.bincount(states_k).tolist()})

model_comparison = pd.DataFrame(model_comparison_rows)
print(model_comparison.to_string(index=False))
print(f"\nBest by AIC: k={model_comparison.loc[model_comparison['AIC'].idxmin(), 'k']} | "
      f"Best by BIC: k={model_comparison.loc[model_comparison['BIC'].idxmin(), 'k']}")


### State sequences at each k

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 4.5), sharex=True)
for ax, k in zip(axes, [2, 3, 4]):
    _, states_k = k_models[k]
    k_cmap = plt.get_cmap('tab10', k)
    ax.imshow(states_k[None, :], aspect='auto', cmap=k_cmap, vmin=0, vmax=k - 1)
    ax.set_yticks([])
    ax.set_title(f'k={k}', fontsize=9, loc='left')
axes[-1].set_xlabel('Trial (within session)')
plt.suptitle(f'Session {EXAMPLE_SESSION[:8]}... - state sequence by k (n_components=18)', fontsize=10)
plt.tight_layout()
plt.show()


### Syllable fingerprints at k=3 and k=4

In [ ]:
for k in [3, 4]:
    _, states_k = k_models[k]
    state_by_sample_k = dict(zip(trial_samples, states_k))
    seq_labeled_k = session_seq[session_seq['sample'].isin(state_by_sample_k)].copy()
    seq_labeled_k['hmm_state'] = seq_labeled_k['sample'].map(state_by_sample_k)

    fig, axes = plt.subplots(1, k, figsize=(4.5 * k, 6), sharex=True)
    for ax, state in zip(axes, range(k)):
        mat = build_mouse_seq(seq_labeled_k[seq_labeled_k['hmm_state'] == state])
        if mat is None:
            ax.axis('off')
            continue
        ax.imshow(mat, aspect='auto', cmap=paw_state_palette, vmin=0, vmax=31, interpolation=None)
        for x in (10, 20, 30):
            ax.axvline(x=x, color='k', linestyle='--', linewidth=0.5)
        ax.set_title(f'State {state} (n={mat.shape[0]})', fontsize=9)
        ax.set_xticks([10, 20, 30])
        ax.set_xticklabels(['Quiescence', 'Stimulus', 'Response'], rotation=30, ha='right', fontsize=7)
    axes[0].set_ylabel('Trials (sorted)')
    plt.suptitle(f'k={k} - syllable fingerprint by state (n_components=18)', fontsize=10)
    plt.tight_layout()
    plt.show()


## Notes / next steps

- This is one arbitrarily-chosen session (not cherry-picked), purely to validate the mechanics: global PCA -> per-session projection -> restart-fitted Gaussian HMM -> visualization all work end to end.
- 30 PCA components (~53% variance) is a placeholder tuned for a single session's sample size, not a principled choice - revisit once scaling to multiple sessions, where more trials could support more components without overfitting.
- Before scaling up: check whether the 2 states here look at all behaviorally meaningful (e.g. relate to this session's own engagement labels, the way `cat-HMM.ipynb` did) - a single session can't establish that on its own, but a qualitative check is cheap.
- If this looks promising, next step is running it across many sessions with the *same* global PCA transform (already session-agnostic by construction here), then reusing the psychometric/chronometric/fingerprint pooling machinery already built.